In [7]:
import lightgbm as lgb
import polars as pl
import polars.selectors as cs
from polars import DataFrame
from sklearn.datasets import fetch_20newsgroups

from polars_ml import Pipeline
from polars_ml.model_selection import KFold, train_test_split
from polars_ml.model_selection.metrics import evaluate_classification_metrics

data, target = fetch_20newsgroups(return_X_y=True)
df = DataFrame({"text": data, "target": target})
df

text,target
str,i64
"""From: lerxst@wam.umd.edu (wher…",7
"""From: guykuo@carson.u.washingt…",4
"""From: twillis@ec.ecn.purdue.ed…",4
"""From: jgreen@amber (Joe Green)…",1
"""From: jcm@head-cfa.harvard.edu…",14
…,…
"""From: jim.zisfein@factory.com …",13
"""From: ebodin@pearl.tufts.edu S…",4
"""From: westes@netcom.com (Will …",3


In [2]:
with pl.Config(
    set_tbl_rows=20, set_fmt_str_lengths=1000, set_fmt_table_cell_list_len=1000
):
    # n-grams
    display(
        df.lazy()
        .select(pl.col("text").str.split(""))
        .with_row_index("row_index")
        .explode("text")
        .with_columns(pl.cum_count("row_index").over("row_index").alias("index"))
        .rolling("index", period="3i", group_by="row_index", offset="-1i")
        .agg("text")
        .with_columns(pl.col("text").list.join("").alias("text"))
        .group_by("text")
        .len()
        .sort("len", descending=True)
        .collect()
    )

text,len
str,u32
""" """,486643
""" th""",229942
"""---""",202857
"""the""",185141
"""he """,143481
"""ing""",89084
""" to""",76920
""" an""",75792
"""to """,71198


In [3]:
with pl.Config(
    set_tbl_rows=20, set_fmt_str_lengths=1000, set_fmt_table_cell_list_len=1000
):
    display(
        df.lazy()
        .select(pl.col("text").str.split(" "))
        .explode("text")
        .group_by("text")
        .len()
        .sort("len", descending=not True)
        .collect()
    )

text,len
str,u32
"""$315 My""",1
"""norris@mit.edu Seeking""",1
"""91109 """,1
"""universal supreme""",1
"""weak. > >But""",1
"""to provide """,1
"""prosperity.""",1
"""(pronounce""",1
"""to: Michael""",1


In [8]:
metrics = []
for train_idx, valid_idx in KFold(
    n_splits=5, shuffle=True, seed=42, stratify="target"
).split(df):
    train_df = df.select(pl.all().gather(train_idx))
    valid_df = df.select(pl.all().gather(valid_idx))

    model = (
        Pipeline()
        .text.tfidf_truncated_svd(
            "text", out_dir="tfidf_truncated_svd", svd_kwargs={"n_components": 100}
        )
        .drop("text")
        .gbdt.lightgbm(
            pl.exclude("target"),
            "target",
            {"objective": "multiclass", "num_class": 20},
            train_kwargs={"callbacks": [lgb.early_stopping(10)]},
        )
        .horizontal.argmax(cs.starts_with("lightgbm_"), value_name="lightgbm")
        .with_columns(
            pl.col("lightgbm")
            .list.first()
            .str.extract(r"lightgbm_(\d+)")
            .cast(pl.UInt8)
        )
    )

    model.fit(train_df, valid_df)
    valid_pred_df = model.transform(valid_df)
    metrics.append(
        evaluate_classification_metrics(
            valid_pred_df,
            "target",
            y_pred_class="lightgbm",
            y_pred_proba_prefix="lightgbm_",
            n_classes=20,
        )
    )

pl.concat(metrics)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002622 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 9045, number of used features: 100
[LightGBM] [Info] Start training from score -3.159325
[LightGBM] [Info] Start training from score -2.963638
[LightGBM] [Info] Start training from score -2.952988
[LightGBM] [Info] Start training from score -2.952988
[LightGBM] [Info] Start training from score -2.974403
[LightGBM] [Info] Start training from score -2.948760
[LightGBM] [Info] Start training from score -2.961499
[LightGBM] [Info] Start training from score -2.946653
[LightGBM] [Info] Start training from score -2.940357
[LightGBM] [Info] Start training from score -2.942451
[LightGBM] [Info] Start training from score -2.936181
[LightGBM] [Info] Start training from score -2.944550
[LightGBM] [Info] Start training from score -2.952988
[LightGB

accuracy,balanced_accuracy,precision_macro,precision_weighted,recall_macro,recall_weighted,f1_macro,f1_weighted,matthews_corrcoef,cohen_kappa_score,roc_auc_ovr,roc_auc_ovo,log_loss
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.801675,0.797298,0.805051,0.805225,0.797298,0.801675,0.799745,0.802169,0.791212,0.791078,0.984423,0.984258,0.674872
0.774051,0.768932,0.780627,0.780766,0.768932,0.774051,0.772351,0.77524,0.762194,0.76197,0.979743,0.979611,0.780178
0.791519,0.788024,0.797092,0.796805,0.788024,0.791519,0.791055,0.792824,0.78053,0.780384,0.981272,0.981136,0.734564
0.776549,0.770384,0.78574,0.785969,0.770384,0.776549,0.775105,0.778561,0.764868,0.764582,0.978876,0.978625,0.7837
0.779157,0.772988,0.783035,0.786777,0.772988,0.779157,0.776026,0.781051,0.767597,0.767374,0.980193,0.979889,0.759939
